# Simulated Pairwise Δ pR² — M29 D23 (Prototype)

Loads real M29 D23 anatomy, cell types and spike trains, but **simulates** Δ pR² values
based on three planted connectivity groups:

| Group | Definition | Expected coordination |
|---|---|---|
| `GC` | Grid cells | Strong internal; moderate with medial NGS |
| `medNGS` | NGS with \|SC_x\| < 3400 µm | Moderate with GC; strong internal |
| `latNGS` | NGS with \|SC_x\| ≥ 3400 µm | Weak cross-region; moderate internal |

**Two target cells are analysed:**
- `test_grid_cell_1 = 214` — a grid cell (co-modular)
- `test_ngs_cell_1  = 340` — a non-grid spatial cell

For each target, three covariate sets are tested:
1. **10 GC / 10 NGS** — a random subset of 10 cells from each type
2. **N_equal GC / N_equal NGS** — all available cells from the rarer type, matched sample from the other

The covariate trace panel shows 10 GC and 10 NGS cells regardless of which set is used in the models.

Y_hat is **simulated** as a weighted blend of the true spike train + mean firing rate, scaled to
match the pR² values expected from the planted connectivity structure.

In [ ]:
import numpy as np
import pandas as pd
import pynapple as nap
import igraph as ig
import leidenalg
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
from scipy.ndimage import gaussian_filter
from matplotlib.colors import LinearSegmentedColormap
from collections import Counter
from spatial_manifolds.detect_grids import *

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

plt.rcParams['font.family'] = 'Arial'

mouse       = 29
day         = 23
source_path = '/Users/harryclark/Downloads/COHORT12/'
fig_path    = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_playground/'
os.makedirs(fig_path, exist_ok=True)

# ── Target cells ──────────────────────────────────────────────────────────────
test_grid_cell_1 = 214   # co-modular grid cell
test_ngs_cell_1  = 340   # non-grid spatial cell

# ── Simulation seeds and parameters ──────────────────────────────────────────
SEED            = 42
ML_BOUNDARY     = 3400          # µm
N_DISPLAY_COV   = 10            # GC and NGS cells shown in the covariate trace panel
N_RANDOM_COV    = 10            # cells per type in the 'random-10' condition
TRACE_WINDOW_VR = (24150, 26250)
TRACE_GAIN      = 5

COL_GC       = '#c04744'
COL_NGS      = '#3171ae'
COL_NGS_LAT  = '#88bbdd'
COL_OTHER    = '#aaaaaa'

# ── Planted Δ pR² means (covariate_group → target_group) ─────────────────────
# Each entry: (mean, std) of Δ pR² drawn from Normal, clipped ≥ 0
PR2_SIM = {
    ('GC',     'GC'):     (0.09, 0.025),
    ('GC',     'medNGS'): (0.06, 0.020),
    ('GC',     'latNGS'): (0.02, 0.015),
    ('medNGS', 'GC'):     (0.05, 0.020),
    ('medNGS', 'medNGS'): (0.07, 0.020),
    ('medNGS', 'latNGS'): (0.01, 0.015),
    ('latNGS', 'GC'):     (0.02, 0.015),
    ('latNGS', 'medNGS'): (0.01, 0.015),
    ('latNGS', 'latNGS'): (0.05, 0.020),
}

def white_to_hex_cmap(hex_color):
    return LinearSegmentedColormap.from_list('cmap', ['#FFFFFF', hex_color])

rng = np.random.default_rng(SEED)

cell_class = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications.csv')
sess_cells = cell_class[
    (cell_class['mouse'] == mouse) & (cell_class['day'] == day)
].copy()
sess_cells['cluster_id'] = sess_cells['cluster_id'].astype(int)
sess_cells['SC_x_abs']   = sess_cells['SC_x'].abs()
print(f'M{mouse} D{day}: {len(sess_cells)} cells')

## 1. Load session data
Load VR spike trains and behaviour for M29 D23. Also classify cells into grid cells (GC)
and non-grid spatial cells (NGS) using `classify_cells_both_sessions`.

In [ ]:
print('Loading VR...')
tcs_vr, tcs_time_vr, _, last_ephys_bin_vr, beh_vr, clusters_vr = compute_vr_tcs(
    mouse, day, apply_zscore=False, apply_guassian_filter=False, source_path=source_path)
last_t_vr = clusters_vr[clusters_vr.index[0]].count(
    bin_size=time_bs, time_units='ms').index[-1]
ep_vr = nap.IntervalSet(start=0, end=last_t_vr, time_units='s')

gcs, ngs, all_cells = classify_cells_both_sessions(mouse, day, source_path=source_path)
gc_ids  = set(gcs.cluster_id.values.astype(int))
ngs_ids = set(ngs.cluster_id.values.astype(int))

# VR position and speed
def _bin(key, T):
    a = np.array(beh_vr[key].bin_average(bin_size=time_bs, time_units='ms', ep=ep_vr))
    return pd.Series(a).ffill().bfill().values[:T]

T_sess = len(np.array(tcs_time_vr[test_grid_cell_1]))
dt     = _bin('travel', T_sess) - ((beh_vr['trial_number'][0] - 1) * tl)
pos_vr = dt % tl
spd_vr = _bin('S', T_sess)

# Confirm both target cells exist
for tid, label in [(test_grid_cell_1, 'GC'), (test_ngs_cell_1, 'NGS')]:
    in_gc  = tid in gc_ids
    in_ngs = tid in ngs_ids
    in_vr  = tid in tcs_time_vr
    print(f'  target {tid} ({label}): in_gc={in_gc}  in_ngs={in_ngs}  in_vr={in_vr}')

print(f'\nVR: {len(tcs_time_vr)} cells  |  GC: {len(gc_ids)}  NGS: {len(ngs_ids)}')

## 2. Assign connectivity groups
Each cell is labelled as `GC`, `medNGS` or `latNGS` based on cell type and mediolateral position
relative to the 3400 µm boundary. Only cells present in the VR recording are included.

In [ ]:
sc_x_lookup = sess_cells.set_index('cluster_id')['SC_x_abs'].to_dict()

def get_group(cid):
    if cid in gc_ids:  return 'GC'
    if cid in ngs_ids: return 'medNGS' if sc_x_lookup.get(cid, 9999) < ML_BOUNDARY else 'latNGS'
    return None

valid_ids = [
    cid for cid in sorted(gc_ids | ngs_ids)
    if cid in tcs_time_vr and cid in sc_x_lookup
]
groups = {cid: get_group(cid) for cid in valid_ids}
groups = {k: v for k, v in groups.items() if v}
valid_ids = list(groups.keys())
N = len(valid_ids)

GROUP_COLOR = {'GC': COL_GC, 'medNGS': COL_NGS, 'latNGS': COL_NGS_LAT}
GROUP_LABEL = {
    'GC':     f'Grid cells (n={sum(v=="GC" for v in groups.values())})',
    'medNGS': f'Medial NGS <{ML_BOUNDARY}µm (n={sum(v=="medNGS" for v in groups.values())})',
    'latNGS': f'Lateral NGS ≥{ML_BOUNDARY}µm (n={sum(v=="latNGS" for v in groups.values())})',
}
print(f'\n{N} cells in analysis:  {Counter(groups.values())}')
print(f'Target {test_grid_cell_1} group: {groups.get(test_grid_cell_1, "NOT FOUND")}')
print(f'Target {test_ngs_cell_1} group: {groups.get(test_ngs_cell_1, "NOT FOUND")}')

fig, ax = plt.subplots(figsize=(5, 3.5))
for grp, col in GROUP_COLOR.items():
    sub = sess_cells[sess_cells['cluster_id'].isin([c for c,g in groups.items() if g==grp])]
    ax.scatter(sub['SC_x_abs'], sub['SC_y'], color=col, s=18, alpha=0.8,
               edgecolors='k', lw=0.3, label=GROUP_LABEL[grp])
for tid in [test_grid_cell_1, test_ngs_cell_1]:
    r = sess_cells[sess_cells['cluster_id'] == tid]
    if len(r):
        ax.scatter(r['SC_x_abs'], r['SC_y'], marker='*', s=200,
                   color='gold', edgecolors='black', lw=0.8, zorder=5,
                   label=f'Target {tid}')
ax.axvline(ML_BOUNDARY, color='black', lw=1.2, ls='--')
ax.invert_yaxis()
ax.legend(fontsize=7, frameon=False, loc='lower right')
ax.set_xlabel('|SC_x| ML (µm)', fontsize=9); ax.set_ylabel('SC_y DV (µm)', fontsize=9)
ax.set_title('Connectivity group assignments', fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 3. Simulate pairwise Δ pR² matrix
Each directed pair (covariate → target) is assigned a Δ pR² drawn from
`Normal(mean, std)` clipped to ≥ 0, where the mean and std are set by the
planted connectivity structure in `PR2_SIM`. The matrix is then symmetrised
by averaging both directions.

In [ ]:
id_to_idx = {cid: i for i, cid in enumerate(valid_ids)}
mat = np.zeros((N, N))

for i, ci in enumerate(valid_ids):
    for j, cj in enumerate(valid_ids):
        if i == j: continue
        key = (groups[ci], groups[cj])
        mu, sig = PR2_SIM.get(key, (0.005, 0.010))
        mat[i, j] = max(0.0, rng.normal(mu, sig))

mat_sym = (mat + mat.T) / 2
np.fill_diagonal(mat_sym, 0)

# Sort by group then ML position for display
_order  = sorted(range(N), key=lambda k: (groups[valid_ids[k]], sc_x_lookup[valid_ids[k]]))
_sorted = mat_sym[np.ix_(_order, _order)]
_glbls  = [groups[valid_ids[k]] for k in _order]

fig, ax = plt.subplots(figsize=(6, 5))
vmax = np.percentile(mat_sym, 98)
im = ax.imshow(_sorted, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
               aspect='auto', interpolation='nearest')
plt.colorbar(im, ax=ax, fraction=0.04).set_label('Simulated Δ pR²', fontsize=8)
prev = None
for k, g in enumerate(_glbls):
    if g != prev:
        ax.axhline(k - 0.5, color='black', lw=0.8)
        ax.axvline(k - 0.5, color='black', lw=0.8)
    prev = g
ax.set_title('Simulated Δ pR² (sorted by group + ML)', fontsize=9)
ax.set_xlabel('Covariate cell'); ax.set_ylabel('Target cell')
plt.tight_layout(); plt.show()

## 4. Leiden community detection
Leiden CPM is run on the symmetrised Δ pR² matrix with a resolution parameter search
(0.5–1.75). We then compare the detected ensembles to the planted groups to assess
whether the simulated connectivity structure is recoverable.

In [ ]:
def run_leiden(sym_mat, n_search=300, res_min=0.5, res_max=1.75,
               n_iter_s=10, n_iter_f=500, seed=42):
    n = sym_mat.shape[0]
    rows, cols = np.triu_indices(n, k=1)
    g = ig.Graph(n=n, edges=list(zip(rows.tolist(), cols.tolist())),
                 directed=False,
                 edge_attrs={'weight': sym_mat[rows, cols].tolist()})
    resolutions = np.linspace(res_min, res_max, n_search)
    best_mod, best_res, mod_curve = -np.inf, resolutions[0], []
    for res in resolutions:
        p = leidenalg.find_partition(g, leidenalg.CPMVertexPartition,
                                     weights='weight', resolution_parameter=res,
                                     n_iterations=n_iter_s, seed=seed)
        mod_curve.append(p.modularity)
        if p.modularity > best_mod: best_mod, best_res = p.modularity, res
    part = leidenalg.find_partition(g, leidenalg.CPMVertexPartition,
                                    weights='weight', resolution_parameter=best_res,
                                    n_iterations=n_iter_f, seed=seed)
    labels = np.array(part.membership)
    for u, c in zip(*np.unique(labels, return_counts=True)):
        if c < 2: labels[labels == u] = -1
    return labels, best_res, best_mod, resolutions, np.array(mod_curve)

print('Running Leiden...')
labels, best_res, best_mod, res_curve, mod_curve = run_leiden(mat_sym)
n_ens = len(np.unique(labels[labels >= 0]))
print(f'{n_ens} ensembles  γ={best_res:.3f}  Q={best_mod:.3f}')

ens_pal = plt.cm.tab20.colors
def ens_color(e): return ens_pal[e % len(ens_pal)] if e >= 0 else '#cccccc'

df_sim = pd.DataFrame({'cluster_id': valid_ids, 'ensemble': labels,
                       'planted': [groups[c] for c in valid_ids]})
df_sim = df_sim.merge(sess_cells[['cluster_id','probe_x','probe_y','SC_x_abs','SC_y']],
                      on='cluster_id', how='left')

print('\nConfusion (planted → Leiden):')
print(pd.crosstab(df_sim['planted'], df_sim['ensemble'],
                  rownames=['Planted'], colnames=['Leiden']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={'wspace': 0.35})

# Sorted Δ pR² matrix with Leiden boundaries
ax = axes[0]
lorder  = np.argsort(labels)
mat_l_s = mat_sym[np.ix_(lorder, lorder)]
lbl_s   = labels[lorder]
vmax    = np.percentile(mat_l_s, 98)
im = ax.imshow(mat_l_s, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
               aspect='auto', interpolation='nearest')
plt.colorbar(im, ax=ax, fraction=0.04).set_label('Δ pR²', fontsize=8)
for b in np.where(np.diff(lbl_s) != 0)[0] + 0.5:
    ax.axhline(b, color='black', lw=0.7); ax.axvline(b, color='black', lw=0.7)
ax.set_title(f'Sorted Δ pR² ({n_ens} Leiden ensembles)', fontsize=9)

# Planted groups in anatomy
ax = axes[1]
for grp, col in GROUP_COLOR.items():
    sub = df_sim[df_sim['planted'] == grp]
    ax.scatter(sub['SC_x_abs'], sub['SC_y'], color=col, s=18, alpha=0.8,
               edgecolors='k', lw=0.3, label=GROUP_LABEL[grp])
for tid in [test_grid_cell_1, test_ngs_cell_1]:
    r = df_sim[df_sim['cluster_id'] == tid]
    if len(r): ax.scatter(r['SC_x_abs'], r['SC_y'], marker='*', s=200,
                          color='gold', edgecolors='k', lw=0.8, zorder=5)
ax.axvline(ML_BOUNDARY, color='black', lw=1.2, ls='--')
ax.invert_yaxis(); ax.set_title('Planted groups', fontsize=9)
ax.legend(fontsize=6, frameon=False); ax.spines[['top','right']].set_visible(False)
ax.set_xlabel('|SC_x| ML (µm)', fontsize=8); ax.set_ylabel('SC_y DV (µm)', fontsize=8)

# Leiden ensembles in anatomy
ax = axes[2]
for ens in sorted(df_sim[df_sim['ensemble'] >= 0]['ensemble'].unique()):
    sub = df_sim[df_sim['ensemble'] == ens]
    ax.scatter(sub['SC_x_abs'], sub['SC_y'], color=ens_color(ens), s=18,
               alpha=0.8, edgecolors='k', lw=0.3, label=f'E{ens} (n={len(sub)})')
for tid in [test_grid_cell_1, test_ngs_cell_1]:
    r = df_sim[df_sim['cluster_id'] == tid]
    if len(r): ax.scatter(r['SC_x_abs'], r['SC_y'], marker='*', s=200,
                          color='gold', edgecolors='k', lw=0.8, zorder=5)
ax.axvline(ML_BOUNDARY, color='black', lw=1.2, ls='--')
ax.invert_yaxis(); ax.set_title('Leiden ensembles', fontsize=9)
ax.legend(fontsize=6, frameon=False, ncol=2); ax.spines[['top','right']].set_visible(False)
ax.set_xlabel('|SC_x| ML (µm)', fontsize=8); ax.set_ylabel('SC_y DV (µm)', fontsize=8)

plt.savefig(fig_path + f'simulated_leiden_M{mouse}D{day}.pdf', bbox_inches='tight', dpi=200)
plt.show()

## 5. Select covariate cells
For each target cell, we build **two covariate sets** using GC and NGS cells:

1. **Random-10**: a random sample of 10 GC cells and 10 NGS cells (excluding the target)
2. **N-equal**: all available cells of the rarer type, with a matched random sample from
   the other type — ensuring both populations contribute the same number of covariate cells

The covariate **trace panel** always shows N_DISPLAY_COV (10) cells from each type
regardless of which model set is plotted.

In [ ]:
TARGET_CELLS = {
    'GC':  test_grid_cell_1,
    'NGS': test_ngs_cell_1,
}

# Available covariate pools (excluding both target cells from both pools)
exclude = {test_grid_cell_1, test_ngs_cell_1}
pool_gc  = sorted([c for c in gc_ids  if c in tcs_time_vr and c not in exclude])
pool_ngs = sorted([c for c in ngs_ids if c in tcs_time_vr and c not in exclude])

N_equal = min(len(pool_gc), len(pool_ngs))
print(f'Available GC covariate cells:  {len(pool_gc)}')
print(f'Available NGS covariate cells: {len(pool_ngs)}')
print(f'N_equal (balanced set):         {N_equal}')
print(f'N_random (random-10 set):        {N_RANDOM_COV}')

# Random-10 selection (same for both targets)
rand_gc  = list(rng.choice(pool_gc,  size=min(N_RANDOM_COV, len(pool_gc)),  replace=False).astype(int))
rand_ngs = list(rng.choice(pool_ngs, size=min(N_RANDOM_COV, len(pool_ngs)), replace=False).astype(int))

# N-equal selection
# If GC is the rarer pool: use all GC, sample N_equal NGS
# If NGS is the rarer pool: use all NGS, sample N_equal GC
if len(pool_gc) <= len(pool_ngs):
    equal_gc  = pool_gc.copy()
    equal_ngs = list(rng.choice(pool_ngs, size=N_equal, replace=False).astype(int))
    print(f'Rarer type: GC  → use all {len(equal_gc)} GC, sample {len(equal_ngs)} NGS')
else:
    equal_ngs = pool_ngs.copy()
    equal_gc  = list(rng.choice(pool_gc, size=N_equal, replace=False).astype(int))
    print(f'Rarer type: NGS → use all {len(equal_ngs)} NGS, sample {len(equal_gc)} GC')

# Display subset (10 of each for the trace panel)
disp_gc  = list(rng.choice(pool_gc,  size=min(N_DISPLAY_COV, len(pool_gc)),  replace=False).astype(int))
disp_ngs = list(rng.choice(pool_ngs, size=min(N_DISPLAY_COV, len(pool_ngs)), replace=False).astype(int))

## 6. Simulate predicted spike trains
For each model condition, Y_hat is simulated as:
`Y_hat = α × y_true + (1−α) × mean(y) + ε`
where α = √(target_pR²) and ε is Gaussian noise. This preserves the spatial
structure of the true rate map at the correct signal-to-noise level.

The expected pR² for each model is set based on the planted connectivity structure —
e.g. GC covariates give higher pR² for a GC target than NGS covariates do,
and N-equal covariates give higher pR² than random-10 covariates (more cells = better prediction).

In [ ]:
def simulate_yhat(y, target_pr2, seed=None):
    rng_l = np.random.default_rng(seed)
    alpha  = np.sqrt(max(target_pr2, 0))
    base   = alpha * y + (1 - alpha) * y.mean()
    return np.maximum(base + rng_l.normal(0, 0.5 * y.std(), size=len(y)), 0)

# pR² targets: (target_group, covariate_type) → simulated pR²
# Scaled by n_cells: more cells → better prediction, logarithmically
def scale_pr2(base_pr2, n, n_ref=10):
    """More covariate cells give higher pR², log-scaled relative to n_ref."""
    return base_pr2 * (1 + 0.25 * np.log(n / n_ref)) if n > 0 else 0

# Per-target model definitions
# Format: (label, description, target_pr2)
def get_model_spec(target_id, target_grp):
    n_r  = len(rand_gc)    # random-10 count
    n_eq = N_equal         # equal count
    gc_mu, _   = PR2_SIM.get(('GC',     target_grp), (0.02, 0.01))
    ngs_mu, _  = PR2_SIM.get(('medNGS', target_grp), (0.02, 0.01))
    return [
        ('null',              'Null',                            0.001),
        ('pos',               'Pos',                             0.09 if target_grp=='GC' else 0.06),
        ('pos+spd+lfp',       'P+S+LFP',                        0.12 if target_grp=='GC' else 0.08),
        ('pos+rand10_gc',     f'P + {n_r} GC (random)',         scale_pr2(gc_mu,  n_r,  10)),
        ('pos+rand10_ngs',    f'P + {n_r} NGS (random)',        scale_pr2(ngs_mu, n_r,  10)),
        ('pos+equal_gc',      f'P + {n_eq} GC (N-equal)',       scale_pr2(gc_mu,  n_eq, 10)),
        ('pos+equal_ngs',     f'P + {n_eq} NGS (N-equal)',      scale_pr2(ngs_mu, n_eq, 10)),
    ]

all_models = {}
for tname, tid in TARGET_CELLS.items():
    tgrp = groups.get(tid, 'GC')
    y    = np.array(tcs_time_vr[tid])
    spec = get_model_spec(tid, tgrp)
    all_models[tname] = {}
    print(f'\n── Target {tid} ({tname}, {tgrp}) ──')
    for label, desc, pr2 in spec:
        all_models[tname][label] = {
            'Y_hat': simulate_yhat(y, pr2, seed=hash(label) % 9999),
            'pR2':   pr2, 'desc': desc
        }
        print(f'  {label:25s}  pR²={pr2:.3f}  ({desc})')

## 7. Reconstruct VR rate maps from predicted spike trains
Uses `compute_vr_tcs_using_expected_spikes` to bin the simulated Y_hat traces
into proper trial × position rate maps. Pseudo cluster IDs are borrowed from
cells not involved in the analysis.

In [ ]:
_pseudo_pool = [c for c in all_cells.cluster_id.values.astype(int)
                if c not in exclude and c not in set(equal_gc) | set(equal_ngs)]

tcs_recon_all = {}   # tcs_recon_all[tname][label] = smoothed rate map
sigma_v = 2.5

for tname, tid in TARGET_CELLS.items():
    print(f'Reconstructing rate maps for target {tid} ({tname})...')
    spec   = get_model_spec(tid, groups.get(tid, 'GC'))
    labels_model = [s[0] for s in spec]
    expected = {
        _pseudo_pool[j]: all_models[tname][lbl]['Y_hat']
        for j, lbl in enumerate(labels_model)
    }
    tcs_r, _, _, last_bin, _, _ = compute_vr_tcs_using_expected_spikes(
        mouse, day, apply_zscore=False, vr_type='VR',
        source_path=source_path, expected_spikes=expected)
    tcs_recon_all[tname] = {
        lbl: gaussian_filter(
            np.nan_to_num(tcs_r[_pseudo_pool[j]]).astype(np.float64), sigma=sigma_v
        )[:last_bin]
        for j, lbl in enumerate(labels_model)
    }
    # True rate map for this target
    tcs_recon_all[tname]['__true__'] = gaussian_filter(
        np.nan_to_num(tcs_vr[tid]).astype(np.float64), sigma=sigma_v
    )[:last_bin]
    tcs_recon_all[tname]['__last_bin__'] = last_bin

print('Done.')

## 8. Combined figure: covariates, prediction traces and rate maps
Each row is one target cell. Columns from left to right:
1. **Covariate traces** — 10 GC cells (red) and 10 NGS cells (blue) over the trace window
2. **Predicted spike traces** — true trace + simulated Y_hat for each model, with
   Poisson-sampled spike ticks overlaid
3. **True rate map**
4. **Predicted rate maps** for each model condition

In [ ]:
# ── Trace helpers ────────────────────────────────────────────────────────────
def _norm01(arr, win):
    seg = np.array(arr)[win].astype(float)
    lo, hi = np.nanmin(seg), np.nanmax(seg)
    return (seg - lo) / (hi - lo + 1e-10)

def draw_covariates(ax, pos, spd, gc_traces, ngs_traces, window):
    win   = slice(*window)
    t_idx = np.arange(*window)
    h = 1.2; lpad = len(t_idx)*0.30; wfrac = len(t_idx)*0.02
    traces = (
        [(pos, 'Pos',   'black',  1.0, 1.3),
         (spd, 'Speed', '#9b59b6',1.0, 1.3)] +
        [(t, f'GC{i+1}',  COL_GC,  0.75, 0.9) for i,t in enumerate(gc_traces)] +
        [(t, f'NGS{i+1}', COL_NGS, 0.75, 0.9) for i,t in enumerate(ngs_traces)]
    )
    n = len(traces)
    for j,(arr,label,c,a,lw) in enumerate(traces):
        off = h*(n-1-j)
        ax.plot(t_idx, _norm01(arr,win)+off, lw=lw, alpha=a, color=c)
        ax.text(t_idx[0]-wfrac, off+h*0.05, label,
                ha='right', va='bottom', fontsize=4.5, color=c)
    # dividers between pos/spd, GC block, NGS block
    for div in [2, 2+len(gc_traces)]:
        ax.axhline(h*(n-div)-h*0.1, color='lightgray', lw=0.5, ls='--')
    bar_len = int(1000/time_bs)
    bar_y   = -h*0.7
    ax.plot([t_idx[0], t_idx[0]+bar_len], [bar_y,bar_y], 'k-', lw=2)
    ax.text(t_idx[0]+bar_len*0.5, bar_y-h*0.15, '1 s',
            ha='center', va='top', fontsize=5.5)
    ax.set_xlim(t_idx[0]-lpad, t_idx[-1]+len(t_idx)*0.03)
    ax.set_ylim(bar_y-h*0.4, h*n+0.6)
    ax.axis('off')

def draw_predictions(ax, y_true, models_d, show_keys, window, gain, color, seed=0):
    rng_spk = np.random.default_rng(seed)
    win   = slice(*window)
    t_idx = np.arange(*window)
    wfrac = len(t_idx)*0.02
    n     = len(show_keys)
    true  = np.array(y_true)[win]
    peak  = float(np.nanmax(np.abs(true)))
    step  = peak*1.5 if peak > 0 else 1.0
    tick_h = step*0.18
    top   = step*n
    # True
    ax.plot(t_idx, true*2+top, lw=1.2, color=color, alpha=0.5)
    for b in np.where(true>0)[0]:
        yb = true[b]*2+top
        ax.plot([t_idx[b],t_idx[b]], [yb, yb+tick_h], color=color, lw=0.9, alpha=0.9)
    ax.text(t_idx[0]-wfrac, top+step*0.05, 'True',
            ha='right', va='bottom', fontsize=5.5, color=color)
    # Predictions
    for j, key in enumerate(show_keys):
        off  = step*(n-1-j)
        pred = np.maximum(models_d[key]['Y_hat'][win], 0)
        pr2  = models_d[key]['pR2']
        desc = models_d[key]['desc']
        ax.plot(t_idx, pred*gain+off, lw=1.0, alpha=0.55, color=color)
        spk_cnt = rng_spk.poisson(pred)
        for b,cnt in enumerate(spk_cnt):
            for _ in range(cnt):
                yb = pred[b]*gain+off
                ax.plot([t_idx[b],t_idx[b]], [yb, yb+tick_h],
                        color=color, lw=0.8, alpha=0.85)
        ax.text(t_idx[0]-wfrac, off+step*0.05, desc,
                ha='right', va='bottom', fontsize=4.5, color=color)
        ax.text(t_idx[-1], off+step*0.05, f'{pr2:.2f}',
                ha='left', va='bottom', fontsize=4.5, color='dimgray')
    ax.set_xlim(t_idx[0]-len(t_idx)*0.30, t_idx[-1]+len(t_idx)*0.14)
    ax.set_ylim(-step*0.3, top+step*1.1)
    ax.axis('off')

print('Helpers defined.')

In [ ]:
SHOW_MODELS = ['null', 'pos', 'pos+spd+lfp',
               'pos+rand10_gc', 'pos+rand10_ngs',
               'pos+equal_gc',  'pos+equal_ngs']

n_models = len(SHOW_MODELS)
n_rows   = len(TARGET_CELLS)
TARGET_COLOR = {'GC': COL_GC, 'NGS': COL_NGS}

# column layout: cov | gap | traces | gap | true_rm | gap | predicted_rms x n_models
col_w   = [2.2, 0.1, 2.5, 0.2, 1.1, 0.1] + [1.1]*n_models
n_cols  = len(col_w)
row_h   = [1.0]*n_rows

fig = plt.figure(figsize=(sum(col_w)*0.80, sum(row_h)*3.8))
gs  = gridspec.GridSpec(n_rows, n_cols, figure=fig,
                        width_ratios=col_w,
                        height_ratios=row_h,
                        hspace=0.25, wspace=0.10)

gc_display_traces  = [np.array(tcs_time_vr[c]) for c in disp_gc]
ngs_display_traces = [np.array(tcs_time_vr[c]) for c in disp_ngs]

for ri, (tname, tid) in enumerate(TARGET_CELLS.items()):
    color   = TARGET_COLOR[tname]
    y_true  = np.array(tcs_time_vr[tid])
    last_b  = tcs_recon_all[tname]['__last_bin__']
    is_bot  = (ri == n_rows - 1)

    # Col 0: covariate traces
    draw_covariates(fig.add_subplot(gs[ri, 0]),
                    pos_vr, spd_vr,
                    gc_display_traces, ngs_display_traces,
                    TRACE_WINDOW_VR)
    if ri == 0:
        fig.axes[-1].set_title(
            f'Covariates\n({N_DISPLAY_COV} GC + {N_DISPLAY_COV} NGS displayed)',
            fontsize=7.5, pad=4)
    fig.axes[-1].text(-0.02, 0.5, f'{tname} target\n(unit {tid})',
                      transform=fig.axes[-1].transAxes,
                      fontsize=7, color=color, fontweight='bold',
                      va='center', ha='right', rotation=90)

    fig.add_subplot(gs[ri, 1]).axis('off')

    # Col 2: prediction traces
    draw_predictions(fig.add_subplot(gs[ri, 2]),
                     y_true, all_models[tname], SHOW_MODELS,
                     TRACE_WINDOW_VR, TRACE_GAIN, color, seed=ri)
    if ri == 0:
        fig.axes[-1].set_title('Predicted spike traces\n(Poisson-sampled ticks)',
                               fontsize=7.5, pad=4)

    fig.add_subplot(gs[ri, 3]).axis('off')

    # Col 4: true rate map
    ax = fig.add_subplot(gs[ri, 4])
    rm_true = tcs_recon_all[tname]['__true__']
    plot_firing_rate_map(ax, rm_true, bs=bs, tl=tl, p=95,
                         cmap=white_to_hex_cmap(color))
    if ri == 0: ax.set_title('True', fontsize=7.5, fontweight='bold')
    ax.set_ylabel('Trial', fontsize=6, labelpad=2)
    if is_bot: ax.set_xlabel('Pos (cm)', fontsize=6)
    ax.tick_params(labelsize=6)

    fig.add_subplot(gs[ri, 5]).axis('off')

    # Cols 6+: predicted rate maps
    for mi, key in enumerate(SHOW_MODELS):
        ax = fig.add_subplot(gs[ri, mi + 6])
        rm = tcs_recon_all[tname][key]
        plot_firing_rate_map(ax, rm, bs=bs, tl=tl, p=95,
                             cmap=white_to_hex_cmap(color))
        if ri == 0:
            ax.set_title(f'{all_models[tname][key]["desc"]}\npR²={all_models[tname][key]["pR2"]:.2f}',
                         fontsize=6, color=color)
        else:
            ax.set_title(f'pR²={all_models[tname][key]["pR2"]:.2f}', fontsize=6, color=color)
        ax.set_yticks([])
        if is_bot: ax.set_xlabel('Pos (cm)', fontsize=6)
        ax.tick_params(labelsize=6)

fig.suptitle(
    f'M{mouse} D{day} — simulated predictions  '
    f'(random-10 vs N-equal={N_equal} GC/NGS covariates)',
    fontsize=10, fontweight='bold', y=1.01)

savepath_combined = (fig_path + f'simulated_traces_combined_M{mouse}D{day}.pdf')
fig.savefig(savepath_combined, bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved → {savepath_combined}')

## 9. Notes on interpreting the simulation

- **The Leiden clusters** (Section 4) should roughly recover the three planted groups
  (GC / medNGS / latNGS). Any deviation reflects noise in the simulated Δ pR² values
  and the resolution parameter choice.

- **The rate maps** (Section 8) become progressively more accurate as more covariate cells
  are added. The N-equal condition uses more cells than random-10 when available cells
  exceed 10, so the improvement reflects sample size, not signal.

- **GC target vs NGS target**: the GC target (214) should show higher pR² with GC covariates
  than NGS covariates (planted GC↔GC coordination is strong). The NGS target (340) shows
  the reverse pattern depending on whether it is medial or lateral NGS.

- **Adjusting the simulation**: change `PR2_SIM` to test different coordination strengths,
  or change `SEED` to resample the noise. The planted structure is defined entirely in
  the config cell.